# Current Weather Map

Enter a place name, click **Get Weather**, and the notebook will:

- geocode the location with Open-Meteo's geocoding API
- fetch current conditions from the Open-Meteo forecast API
- show a compact weather summary
- render an interactive OpenStreetMap view centered on the result

This notebook uses only the Python standard library plus `ipywidgets` and `IPython.display`, which are already typical in Jupyter environments.

In [1]:
from __future__ import annotations

import json
import urllib.parse
import urllib.request
import uuid
from html import escape

import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

WEATHER_CODES = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Depositing rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    56: "Light freezing drizzle",
    57: "Dense freezing drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    66: "Light freezing rain",
    67: "Heavy freezing rain",
    71: "Slight snow fall",
    73: "Moderate snow fall",
    75: "Heavy snow fall",
    77: "Snow grains",
    80: "Slight rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    85: "Slight snow showers",
    86: "Heavy snow showers",
    95: "Thunderstorm",
    96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail",
}


def fetch_json(url: str) -> dict:
    request = urllib.request.Request(url, headers={"User-Agent": "weather-map-notebook/1.0"})
    with urllib.request.urlopen(request, timeout=20) as response:
        return json.load(response)


def geocode_location(query: str) -> dict:
    params = urllib.parse.urlencode(
        {
            "name": query,
            "count": 1,
            "language": "en",
            "format": "json",
        }
    )
    payload = fetch_json(f"https://geocoding-api.open-meteo.com/v1/search?{params}")
    results = payload.get("results") or []
    if not results:
        raise ValueError(f"No geocoding result found for '{query}'.")

    match = results[0]
    admin_bits = [match.get("admin1"), match.get("country")]
    subtitle = ", ".join(bit for bit in admin_bits if bit)
    display_name = match["name"] if not subtitle else f"{match['name']}, {subtitle}"
    return {
        "name": match["name"],
        "display_name": display_name,
        "latitude": match["latitude"],
        "longitude": match["longitude"],
        "country": match.get("country", ""),
        "timezone": match.get("timezone", ""),
    }


def get_current_weather(latitude: float, longitude: float) -> tuple[dict, dict]:
    params = urllib.parse.urlencode(
        {
            "latitude": latitude,
            "longitude": longitude,
            "current": ",".join(
                [
                    "temperature_2m",
                    "apparent_temperature",
                    "relative_humidity_2m",
                    "precipitation",
                    "weather_code",
                    "cloud_cover",
                    "surface_pressure",
                    "wind_speed_10m",
                    "wind_direction_10m",
                ]
            ),
            "timezone": "auto",
            "forecast_days": 1,
        }
    )
    payload = fetch_json(f"https://api.open-meteo.com/v1/forecast?{params}")
    return payload["current"], payload["current_units"]


def wind_direction_label(degrees: float | None) -> str:
    if degrees is None:
        return "n/a"
    directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    index = int((degrees % 360) / 45 + 0.5) % 8
    return directions[index]


def weather_label(code: int | None) -> str:
    if code is None:
        return "Unknown"
    return WEATHER_CODES.get(code, f"Code {code}")


def summary_html(place: dict, current: dict, units: dict) -> HTML:
    rows = [
        ("Condition", weather_label(current.get("weather_code"))),
        ("Temperature", f"{current.get('temperature_2m')} {units.get('temperature_2m', '')}"),
        ("Feels like", f"{current.get('apparent_temperature')} {units.get('apparent_temperature', '')}"),
        ("Humidity", f"{current.get('relative_humidity_2m')} {units.get('relative_humidity_2m', '')}"),
        ("Wind", f"{current.get('wind_speed_10m')} {units.get('wind_speed_10m', '')} {wind_direction_label(current.get('wind_direction_10m'))}"),
        ("Precipitation", f"{current.get('precipitation')} {units.get('precipitation', '')}"),
        ("Cloud cover", f"{current.get('cloud_cover')} {units.get('cloud_cover', '')}"),
        ("Pressure", f"{current.get('surface_pressure')} {units.get('surface_pressure', '')}"),
        ("Coordinates", f"{place['latitude']:.4f}, {place['longitude']:.4f}"),
        ("Timezone", place.get('timezone', '')),
        ("Observed at", current.get('time', '')),
    ]

    table_rows = "".join(
        f"<tr><th>{escape(label)}</th><td>{escape(str(value))}</td></tr>" for label, value in rows
    )
    return HTML(
        f"""
        <div style="display:grid; grid-template-columns: minmax(260px, 420px); gap: 1rem; margin: 0.75rem 0 1rem 0;">
          <div style="border: 1px solid #d8dee4; border-radius: 12px; padding: 1rem 1.1rem; background: linear-gradient(180deg, #fbfdff 0%, #f3f8fc 100%); box-shadow: 0 1px 4px rgba(0,0,0,0.06);">
            <div style="font-size: 1.4rem; font-weight: 700; margin-bottom: 0.15rem;">{escape(place['display_name'])}</div>
            <div style="color: #4b5563; margin-bottom: 0.75rem;">Current weather snapshot</div>
            <table style="border-collapse: collapse; width: 100%;">
              <tbody>
                {table_rows}
              </tbody>
            </table>
          </div>
        </div>
        <style>
          th, td {{ padding: 0.35rem 0; text-align: left; vertical-align: top; }}
          th {{ width: 9rem; color: #4b5563; font-weight: 600; }}
        </style>
        """
    )


def leaflet_map_html(place: dict) -> HTML:
    map_id = f"weather-map-{uuid.uuid4().hex}"
    label = json.dumps(place["display_name"])
    latitude = place["latitude"]
    longitude = place["longitude"]
    return HTML(
        f"""
        <div id="{map_id}" style="height: 420px; width: 100%; border: 1px solid #d8dee4; border-radius: 12px; overflow: hidden;"></div>
        <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" integrity="sha256-p4NxAoJBhIIN+hmNHrzRCf9tD/miZyoHS5obTRR9BMY=" crossorigin=""/>
        <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js" integrity="sha256-20nQCchB9co0qIjJZRGuk2/Z9VM+kNiyxNV1lvTlZBo=" crossorigin=""></script>
        <script>
        (function() {{
            function initMap() {{
                const el = document.getElementById({json.dumps(map_id)});
                if (!el || el.dataset.ready === "1" || typeof L === "undefined") {{
                    return;
                }}
                el.dataset.ready = "1";
                const map = L.map(el).setView([{latitude}, {longitude}], 10);
                L.tileLayer("https://tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png", {{
                    maxZoom: 19,
                    attribution: "&copy; OpenStreetMap contributors",
                }}).addTo(map);
                L.marker([{latitude}, {longitude}]).addTo(map).bindPopup({label}).openPopup();
            }}
            initMap();
            setTimeout(initMap, 150);
        }})();
        </script>
        """
    )


location_input = widgets.Text(
    value="94041",
    description="Location",
    layout=widgets.Layout(width="420px"),
)
fetch_button = widgets.Button(description="Get Weather", button_style="primary", icon="cloud")
status = widgets.HTML()
output = widgets.Output()


def render_weather(_=None) -> None:
    query = location_input.value.strip()
    if not query:
        status.value = "<span style='color:#b42318'>Enter a location first.</span>"
        return

    status.value = "<span style='color:#475467'>Loading weather data...</span>"
    with output:
        clear_output()
        try:
            place = geocode_location(query)
            current, units = get_current_weather(place["latitude"], place["longitude"])
            status.value = ""
            display(summary_html(place, current, units))
            display(leaflet_map_html(place))
        except Exception as exc:
            status.value = f"<span style='color:#b42318'>Request failed: {escape(str(exc))}</span>"


fetch_button.on_click(render_weather)
display(
    widgets.VBox(
        [
            widgets.HBox([location_input, fetch_button]),
            status,
            output,
        ]
    )
)

render_weather()